# LangGraph, from a Single Node to Two Workflow Patterns

Your two scripts each demonstrate a different **workflow pattern** for orchestrating LLM calls with LangGraph:

- `routing.py` — classify a request, then send it down exactly **one** of several paths (*routing*)
- `parallelization.py` — fire off several **independent** LLM calls at once and merge the results (*parallelization*)

These map directly onto two of the patterns from Anthropic's "Building Effective Agents" writeup on compound LLM systems. You've already built a third one yourself — an evaluator-optimizer loop in LangGraph — so by the end of this notebook you'll have three of the standard patterns under your belt, and you'll see how they all rest on the *same* small set of LangGraph primitives (`StateGraph`, nodes, edges, `compile`).

We won't walk through your scripts top-to-bottom. Instead we'll build up the underlying concepts one at a time, in the order that makes them easiest to understand, and use your real code as the worked examples.

**A note before we start:** these cells are written to be run in your own environment (with `GROQ_API_KEY` set in a `.env` file). This sandbox doesn't have network access to Groq's API, so the code cells below are not executed here — copy the notebook to an environment with your key to actually run them.

## 0. Setup

Both scripts start the same way: pull in LangGraph's graph-building primitives, a Groq-backed chat model, and `python-dotenv` to load your API key from a `.env` file rather than hardcoding it.

`ChatGroq(model="openai/gpt-oss-120b")` — this is one of the OSS models Groq currently serves (you've noted several older Groq models have since been deprecated, so it's worth periodically re-checking what's live).

In [ ]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b")


## 1. State: the one thing every node shares

A LangGraph graph is built around a single **state** object that every node reads from and writes to. Before writing any node, you declare its shape with a `TypedDict`.

Why bother? Two reasons:

1. **A contract.** Every node knows exactly what fields exist and (in principle) what type they hold, without needing to inspect what upstream nodes did.
2. **Tooling.** LangGraph uses the `TypedDict` to know which keys exist when it merges a node's return value into the running state.

One caveat worth internalizing early: a `TypedDict` is a *typing hint*, not a runtime validator. Nothing stops a node from returning a string where you declared a `list`, and nothing stops you from initializing a field with a value that doesn't match a `Literal` type used elsewhere (we'll see this exact thing happen in a moment).

Here's the state for the routing graph — one query in, one classification, one final response, and a running list of "tools" the handler pretended to use:

In [ ]:
class SupportState(TypedDict):
    customer_query: str
    query_category: str
    response: str
    tools_used: list[str]


## 2. A node is just a function

This is the part that makes LangGraph feel lightweight once it clicks: a **node** is a plain Python function that

- takes the current state (a dict matching your `TypedDict`),
- does something (often: calls an LLM),
- returns a **partial** dict of the fields it wants to update.

LangGraph merges that returned dict into the overall state with a shallow update — by default, each key you return simply **overwrites** the old value for that key. You don't have to return the whole state back, only the fields that changed. That's a deliberate design choice: it keeps each node focused on its own slice of the state instead of having to thread the entire object through every function.

The simplest node in `routing.py` is `handle_general` — no branching, no structured output, just a prompt and a return:

In [ ]:
def handle_general(state: SupportState) -> SupportState:
    """Handle General Queries"""
    prompt = f"""You are a general support specialist. Handle this customer query:
    {state["customer_query"]}

    Provide a helpful response that:
    - Answers their question clearly
    - Provides relevant links or resources
    - Offers additional help if needed
    - Suggests related features they might find useful

    Be friendly and informative
"""
    response = llm.invoke(prompt).content
    print("General Handler: Response generated\n")

    return {
        "response": response,
        "tools_used": ["knowledge_database", "faq_database"]
    }


## 3. The smallest possible graph

Before adding any branching logic, let's wire up a one-node graph so the mechanics of `StateGraph` are clear on their own:

- `StateGraph(SupportState)` creates a builder bound to your state schema.
- `add_node(name, fn)` registers a node under a string name (used later to reference it in edges).
- `add_edge(START, "general")` and `add_edge("general", END)` — `START` and `END` are LangGraph's special sentinel nodes marking where execution begins and ends. Every graph needs at least one edge from `START` and at least one path reaching `END`.
- `.compile()` turns the builder into a runnable graph object.
- `.invoke(initial_state_dict)` runs it synchronously and returns the final state.

Note that `invoke` requires you to supply *all* declared keys up front (even as empty strings/lists) — `TypedDict` doesn't give you default values, so the caller has to.

In [ ]:
mini_builder = StateGraph(SupportState)
mini_builder.add_node("general", handle_general)
mini_builder.add_edge(START, "general")
mini_builder.add_edge("general", END)

mini_graph = mini_builder.compile()

mini_result = mini_graph.invoke({
    "customer_query": "What features are included in the pro plan?",
    "query_category": "",
    "response": "",
    "tools_used": []
})
print(mini_result["response"])


## 4. Getting structured data out of an LLM

To route intelligently, we first need the LLM to tell us *which category* a query belongs to — as a value we can branch on in code, not as a sentence we'd have to regex-parse.

The old-school way to do this is prompt engineering plus manual parsing: "respond with just the category name" and then `response.content.strip().lower()`, hoping the model doesn't wrap it in a sentence or add punctuation. That's brittle — you're one "Sure, the category is: billing." away from a broken parse.

The modern approach, used here, is `with_structured_output`:

1. Define the expected shape as a **Pydantic model**.
2. `Literal["billing", "technical", "refund", "general"]` constrains a field to an exact enum of allowed strings — this is what actually enforces "must be one of these four," at both the type-checker level and (via the generated JSON schema) at the model-provider level.
3. `Field(description=...)` isn't just documentation — that description is included in the schema sent to the LLM provider, so it directly shapes what the model produces.
4. `llm.with_structured_output(QueryClassification)` wraps your chat model so that instead of returning a `content` string, `.invoke()` returns an already-parsed instance of `QueryClassification`.

Under the hood this typically rides on the provider's native function-calling / tool-use support (Groq included), which is far more reliable than asking the model to "please output JSON" in prose.

In [ ]:
class QueryClassification(BaseModel):
    category: Literal["billing", "technical", "refund", "general"]
    confidence: float = Field(description="Confidence score 0-1")
    reasoning: str = Field(description="Why this category was chosen?")


Now the classifier node. Notice it creates a *new* wrapped LLM (`llm.with_structured_output(...)`) rather than calling `llm.invoke` directly — the base `llm` object is left untouched so it can still be used for free-text generation elsewhere in the graph.

**⚠️ Bug watch:** the initial state you'll pass into the full graph later sets `"query_category": ""`. But `QueryClassification.category` is typed as `Literal["billing", "technical", "refund", "general"]` — `""` is not a valid member of that literal. This never actually breaks anything at runtime (Python/TypedDict won't enforce it, and `classify_query` immediately overwrites the field before anything reads it), but it's exactly the kind of thing that would fail a real type-checker like `mypy`, and it's worth developing the habit of noticing.

In [ ]:
def classify_query(state: SupportState) -> SupportState:
    print("\nROUTER: Analyzing customer query ...")

    classifier_llm = llm.with_structured_output(QueryClassification)

    prompt = f""" Classify the following customer support query into one of these categories:

    - billing: Questions about invoices, payments, charges, pricing
    - technical: Technical issues, bugs, feature problems, integration issues.
    - refund: Refund requests, returns, cancellations
    - general: General questions, feature inquiries, information requests

    Customer Query: {state['customer_query']}

    Classify accurately based on the primary intent.
"""
    classification = classifier_llm.invoke(prompt)

    print(f"Category: {classification.category}")
    print(f"Confidence: {classification.confidence:.2f}")
    print(f"Reasoning: {classification.reasoning}\n")

    return {
        "query_category": classification.category
    }


## 5. Conditional edges: turning a value into a branch

This is the routing-specific primitive: `add_conditional_edges(source_node, router_fn, mapping)`.

- `source_node` — the node after which the branch decision happens.
- `router_fn` — a plain function that reads the state and **returns a string**.
- `mapping` — a dict from that string to the *actual node name* to jump to next.

The router function's return type is annotated `Literal["billing", "technical", "refund", "general"]` purely for readability/type-checking and for LangGraph's graph-visualization tooling — it isn't what performs the routing. The `mapping` dict is what LangGraph actually uses at runtime to decide where to go. In this script the returned strings and the node names happen to be identical (`"billing" -> "billing"`, etc.), which makes the indirection *look* redundant — but it doesn't have to be a 1:1 identity mapping; you could just as easily return `"needs_billing_team"` and map that string to a node literally named `"billing"`. Keeping router-function output and node names decoupled like this is what lets you rename nodes without touching your classification logic, or map several different classifier outputs onto the same handler node.

**⚠️ Bug watch:** look closely at the print statement below — `f"...{state['query_category']} handler]\n"` has a stray `]` with no matching `[`. Harmless (it only affects a log line), but it's the kind of copy-paste artifact worth cleaning up.

In [ ]:
def route_query(state: SupportState) -> Literal["billing", "technical", "refund", "general"]:
    print(f"ROUTING: Directing to {state['query_category']} handler\n")  # note: original had a stray ']' here

    return state["query_category"]


## 6. The handler nodes

Three more nodes, one per category, each following the same shape as `handle_general`: build a role-specific prompt, call `llm.invoke(...).content`, return `response` plus a `tools_used` list that overwrites whatever was there before (since there's no reducer — more on that later).

**⚠️ Bug watch (copy-paste error):** in `handle_technical`, the print statement says `"TECHNICAL HANDLER: Processing billing query"` — leftover text from `handle_billing`. Purely cosmetic (it doesn't affect what gets sent to the LLM), but worth fixing so your logs aren't misleading during debugging.

**A design note, not a bug:** `handle_billing`, `handle_technical`, `handle_refund`, and `handle_general` are all structurally identical — same shape, different prompt text and different `tools_used` list. Once you have three or more of these, it's often worth collapsing them into one parametrized `make_handler(role, tools)` factory function instead of four near-duplicate node functions. Not required, but it's the kind of refactor that pays off as soon as you add a fifth category.

In [ ]:
def handle_billing(state: SupportState) -> SupportState:
    print("BILLING HANDLER: Processing billing query")

    prompt = f"""You are a billing specialist. Handle this customer query:

    {state['customer_query']}

    Provide a helpful response that:
    - References billing policies and procedures
    - Offers to check their account details
    - Provides clear next steps
    - Mentions relevant payment options

    Keep it professional and reassuring
"""
    response = llm.invoke(prompt).content
    print("BILLING HANDLER: Response Generated")

    return {
        "response": response,
        "tools_used": ["billing_database", "payment_processor"]
    }


def handle_technical(state: SupportState) -> SupportState:
    print("TECHNICAL HANDLER: Processing technical query")  # fixed copy-paste text

    prompt = f""" You are a technical support specialist. Handle this customer query:

    {state['customer_query']}

    Provide a helpful response that:
    - Offers specific troubleshooting steps
    - References relevant documentation
    - Asks clarifying questions if needed
    - Provides a workaround if applicable

    Be clear and technical but accessible
"""
    response = llm.invoke(prompt).content
    print("TECHNICAL HANDLER: Response Generated")

    return {
        "response": response,
        "tools_used": ["knowledge_base", "bug_tracker", "system_logs"]
    }


def handle_refund(state: SupportState) -> SupportState:
    print("REFUND HANDLER: Processing refund query")

    prompt = f""" You are a refund specialist. Handle this customer query:

    {state['customer_query']}

    Provide a helpful response that:
    - Explains the refund policy clearly
    - Outlines the refund process and timeline
    - Asks for necessary information (order number, reason, etc.)
    - Shows empathy and understanding

    Be empathetic and solution-focused.
"""
    response = llm.invoke(prompt).content
    print("REFUND HANDLER: Response Generated")

    return {
        "response": response,
        "tools_used": ["order_database", "refund_processor"]
    }


## 7. Assembling the full routing graph

Now we put it together: five nodes, an entry edge into the classifier, a conditional branch out of it, and every handler funnels straight to `END`.

**⚠️ Bug watch (dead code):** the original script defines `billing_prompt`, `technical_prompt`, and `refund_prompt` right before the `graph.invoke(...)` call, but only ever passes `general_prompt` into the graph. The other three variables are built and never used — classic leftover-from-testing dead code. Not harmful, just noise; either delete them or actually loop over all four to sanity-check every branch (which is arguably what they were there for in the first place — see the exercises).

In [ ]:
builder = StateGraph(SupportState)

builder.add_node("classify_query", classify_query)
builder.add_node("billing", handle_billing)
builder.add_node("technical", handle_technical)
builder.add_node("refund", handle_refund)
builder.add_node("general", handle_general)

builder.add_edge(START, "classify_query")
builder.add_conditional_edges(
    "classify_query",
    route_query,
    {
        "billing": "billing",
        "technical": "technical",
        "refund": "refund",
        "general": "general"
    }
)

builder.add_edge("billing", END)
builder.add_edge("technical", END)
builder.add_edge("refund", END)
builder.add_edge("general", END)

graph = builder.compile()


In [ ]:
general_prompt = "What features are included in the pro-plan"

result = graph.invoke({
    "customer_query": general_prompt,
    "query_category": "",
    "response": "",
    "tools_used": []
})

print("=" * 70)
print(f"FINAL RESPONSE ({result['query_category'].upper()})")
print("=" * 70)
print(result["response"])
print(f"\nTools Used: {', '.join(result['tools_used'])}")


## 8. A different problem: independent tasks, not exclusive ones

Routing answers "given this input, which *one* path should we take?" That's exclusive branching — exactly one handler runs.

`parallelization.py` solves a different problem: generate an Instagram caption, a tweet, *and* a LinkedIn post for the same topic. These three tasks don't depend on each other at all — there's no reason to run them one after another. This is the **parallelization** pattern: fan work out across independent branches, then fan the results back in.

The state shape reflects that — one shared `topic`, and a separate output field per platform so the branches never collide:

In [ ]:
class OverallState(TypedDict):
    topic: str
    instagram_post: str
    twitter_post: str
    linked_in_post: str
    final_output: str


## 9. Fan-out: multiple edges from the same node

Here's the new primitive. Instead of one `add_edge(START, "some_node")`, we add **three**:

```python
builder.add_edge(START, "instagram")
builder.add_edge(START, "twitter")
builder.add_edge(START, "linkedin")
```

Whenever a node (including `START`) has more than one outgoing edge, LangGraph treats those as parallel branches: all three target nodes run within the same "superstep" instead of waiting on each other. You don't write any threading or `asyncio` code yourself — this concurrency comes for free from how the graph is shaped, using `.invoke()` exactly as before. (If you want *true* async concurrency down to the network-call level rather than just topological parallel scheduling, LangGraph also exposes `.ainvoke()` for graphs built from `async def` nodes — worth knowing about even though these scripts don't use it.)

Here are the three generator nodes. Each is structurally the same as `handle_general` from Section 2 — different prompt, different output key.

**⚠️ Bug watch (typos, cosmetic):** `"Insstagram Generator..."` and `"Agregator: Combining..."` (later) both have extra/missing letters in their print statements, and the LinkedIn prompt has `"proffesional development"` baked into the prompt text itself. That last one is worth fixing specifically because, unlike the print typos, it's inside the actual prompt sent to the model — small misspellings in prompts are usually harmless to LLMs but it's good hygiene to keep prompt text clean.

**⚠️ Bug watch (dead code, harmless):** in the original `generate_twitter`, the line is written as:
```python
twitter_post = llm.invoke(prompt).content\
    
    print("Twitter post generated")
```
That trailing backslash is a line-continuation character, but the next physical line is blank (whitespace only), so the continuation ends there and `print(...)` on the following line is parsed as its own independent statement — the code *does* work, but the stray backslash serves no purpose and reads as a mistake. Cleaned up below.

In [ ]:
def generate_instagram(state: OverallState) -> OverallState:
    """Generate an engaging Instagram post with emojis and hashtags"""
    print("Instagram Generator: Creating post ...")

    prompt = f"""
    Create an Instagram post about: {state['topic']}

    Requirements:
    - Engaging and visual language
    - 2-3 short paragraphs (150-200 words max)
    - Include relevant emojis
    - End with 5-8 relevant hashtags
    - Casual, friendly tone
    - Call-to-action to engage with the post

    Make it perfect for instagram audience
    """
    instagram_post = llm.invoke(prompt).content
    print("Instagram Generator Complete")

    return {"instagram_post": instagram_post}


def generate_twitter(state: OverallState) -> OverallState:
    """Generate a concise Twitter post"""
    print("Twitter Generator: Creating post ...")

    prompt = f"""
    Create a Twitter post about: {state['topic']}

    Requirements:
    - Maximum 280 characters (this is crucial!)
    - Punchy and attention-grabbing
    - Include 2-3 relevant hashtags
    - Conversational tone
    - Can use emojis sparingly
    - Should spark engagement/replies

    Make it perfect for Twitter's fast-paced environment.
    """
    twitter_post = llm.invoke(prompt).content
    print("Twitter post generated")

    return {"twitter_post": twitter_post}


def generate_linkedin(state: OverallState) -> OverallState:
    """Generate a professional LinkedIn post"""
    print("LinkedIn Generator: Creating post ...")

    prompt = f"""Create a linkedin post about: {state['topic']}

    Requirements:
    - Professional yet engaging tone
    - 3-4 paragraphs (200-300 words)
    - Include insights or lessons learned
    - Use line breaks for readability
    - Add 3-5 professional hashtags
    - Include a thought-provoking question at the end.
    - Focus on value and professional development.

    Make it perfect for LinkedIn's professional audience."""
    linkedin_post = llm.invoke(prompt).content
    print("LinkedIn Generator Complete\n")

    return {"linked_in_post": linkedin_post}


## 10. Fan-in: the join is implicit

Fan-out needed an explicit primitive (multiple `add_edge` calls). Fan-in doesn't — it falls out for free from how edges work. `aggregate_posts` has **three incoming edges** (`instagram -> aggregate`, `twitter -> aggregate`, `linkedin -> aggregate`). LangGraph automatically waits for *all* of a node's incoming edges to have fired before running it — so `aggregate` acts as a synchronization barrier without you writing any barrier/join logic yourself. This is the mechanism that makes "fan out then merge" a first-class pattern rather than something you'd hand-roll with threads and locks.

**⚠️ Bug watch (cosmetic, but affects real output):** `aggregate_posts` builds `final_output` as a triple-quoted f-string, and because the `return` statement is indented inside the function, every line of that string carries the function's indentation as literal leading whitespace. It won't error, but if you print or save `final_output`, every line will have a few extra leading spaces baked in — often invisible in a terminal, but visible (and annoying) if you ever write it to a file and expect clean left-aligned text. The fix is either to dedent the triple-quoted string to column 0 regardless of the surrounding code's indentation, or run it through `textwrap.dedent(...)` before returning.

In [ ]:
def aggregate_posts(state: OverallState) -> OverallState:
    """Combine all platform posts into a formatted final output"""
    print("Aggregator: Combining all posts...\n")

    final_output = f"""{"=" * 70}
Social Media Content Package
{"=" * 70}
Topic: {state['topic']}
{"=" * 70}
Instagram Post:
{"=" * 70}
{state['instagram_post']}
{"=" * 70}
LinkedIn Post:
{"=" * 70}
{state["linked_in_post"]}
{"=" * 70}
Twitter Post:
{"=" * 70}
{state['twitter_post']}
{"=" * 70}"""

    return {"final_output": final_output}


## 11. Assembling the parallelization graph

Same building blocks as before — `add_node`, `add_edge`, `compile` — just arranged in a fan-out/fan-in shape instead of a linear or branching one.

In [ ]:
builder = StateGraph(OverallState)

builder.add_node("instagram", generate_instagram)
builder.add_node("twitter", generate_twitter)
builder.add_node("linkedin", generate_linkedin)
builder.add_node("aggregate", aggregate_posts)

builder.add_edge(START, "linkedin")
builder.add_edge(START, "twitter")
builder.add_edge(START, "instagram")

builder.add_edge("linkedin", "aggregate")
builder.add_edge("twitter", "aggregate")
builder.add_edge("instagram", "aggregate")

builder.add_edge("aggregate", END)

graph = builder.compile()


In [ ]:
topic = "The impact of Calisthenics on Body"

result = graph.invoke({
    "topic": topic,
    "instagram_post": "",
    "twitter_post": "",
    "linked_in_post": "",
    "final_output": ""
})

print(result["final_output"])


## 12. Why the parallel writes never collide

Worth naming explicitly, because it's the thing that makes fan-out *safe* here: `instagram`, `twitter`, and `linkedin` each return a dict touching a **different** key (`instagram_post`, `twitter_post`, `linked_in_post`). Since the default state-update behavior is "overwrite this key with whatever the node returned," three branches writing to three different keys never step on each other — LangGraph just merges all three partial updates into the shared state once the superstep finishes.

If two *concurrent* nodes both tried to write to the **same** key in the same superstep, LangGraph would raise an `InvalidUpdateError` — because it has no way to know whether you meant "the second one should win" or "these should be combined somehow." The fix, if you ever need it, is to declare that field with an explicit **reducer** using `Annotated`, e.g.:

```python
from typing import Annotated
import operator

class State(TypedDict):
    tools_used: Annotated[list[str], operator.add]
```

That tells LangGraph *how* to combine multiple writes to `tools_used` (concatenate the lists) instead of just picking a winner. Neither of your two scripts needs this — but it's the concept that sits directly behind the safety of the fan-out pattern, and it's the same idea (state fields "accumulating" instead of being overwritten) you may recognize from working with message-history reducers in chatbot-style graphs.

## 13. Putting the two patterns side by side

| | Routing | Parallelization |
|---|---|---|
| **Question it answers** | "Which *one* path fits this input?" | "How do I run several *independent* tasks at once?" |
| **Key primitive** | `add_conditional_edges` (one branch taken) | multiple `add_edge` calls from one source (all branches taken) |
| **Number of nodes that actually run** | exactly 1 of N handlers | all N branches, every time |
| **Join** | none needed — each branch goes straight to `END` | implicit — a node with multiple incoming edges waits for all of them |
| **State-key discipline** | handlers can share `response` (only one ever runs) | branches must write to *different* keys, or need a reducer |

Both patterns are instances of the same underlying idea: a **workflow** is a fixed, predetermined structure of LLM calls, as opposed to an **agent**, which decides its own next step at runtime. You've already built a third structural pattern in LangGraph — the **evaluator-optimizer loop** you debugged recently (generator produces something, evaluator critiques it, loop back until it passes) — which is a third fixed structure, just with a cycle in the graph instead of a branch or a fan-out. Routing, parallelization, and evaluator-optimizer are three of the small set of composable shapes that most production LLM pipelines turn out to be built from; the classifier + structured-output work you did here is also exactly the same technique you'd use to build the "evaluator" half of an evaluator-optimizer graph, just applied to a different decision.

## Exercises

1. **Fix the flagged bugs.** Go through both scripts and correct every item flagged above (the stray `]`, the copy-pasted "billing query" log line, the `proffesional`/`Insstagram`/`Agregator` typos, the unused `billing_prompt`/`technical_prompt`/`refund_prompt` variables, and the indentation issue in `aggregate_posts`).

2. **Exercise every branch.** Instead of only invoking `graph.invoke(...)` once with `general_prompt`, loop over `billing_prompt`, `technical_prompt`, `refund_prompt`, and `general_prompt` and confirm the classifier routes each one to the handler you'd expect. This is the natural use for those otherwise-dead variables.

3. **Add a fifth routing category.** Add `"escalation"` to the `Literal` in `QueryClassification`, write a `handle_escalation` node, register it, and add the matching entry to the routing dict passed into `add_conditional_edges`. This exercises the full routing wiring end to end.

4. **Make `tools_used` accumulate instead of overwrite.** Change it to `Annotated[list[str], operator.add]` in `SupportState`, and have `classify_query` also return a `tools_used` list (e.g. `["classifier"]`). Confirm the final `tools_used` in the result now contains entries from *both* `classify_query` and whichever handler ran, instead of the handler's list clobbering the classifier's.

5. **Extend the fan-out.** Add a fourth platform (e.g. `generate_facebook`) to the parallelization graph: write the node, add `add_edge(START, "facebook")`, add `add_edge("facebook", "aggregate")`, and update `aggregate_posts` to include it in `final_output`.

6. **Try real async concurrency.** Rewrite the three generator nodes as `async def` functions using `await llm.ainvoke(prompt)`, and call the compiled graph with `await graph.ainvoke(...)` instead of `.invoke(...)`. Compare wall-clock time against the sync version (with three real network calls, the difference should be noticeable).

## Summary — concepts covered

- **`StateGraph` + `TypedDict`** — declaring a shared, typed state schema that every node reads and writes.
- **Nodes as functions** — take the full state, return a *partial* dict of updates; LangGraph merges it in.
- **`START` / `END`** — sentinel nodes marking graph entry and exit.
- **`add_edge` / `.compile()` / `.invoke()`** — the minimal machinery to build and run a graph.
- **Structured output** (`BaseModel`, `Literal`, `Field`, `with_structured_output`) — getting a typed, machine-usable value out of an LLM instead of parsing free text.
- **`add_conditional_edges`** — routing: a router function returns a string, a mapping dict sends execution to exactly one node.
- **Fan-out** — multiple `add_edge` calls from one source node run their targets as parallel branches within a superstep.
- **Fan-in (implicit join)** — a node with multiple incoming edges automatically waits for all of them before running.
- **Default state-update behavior (overwrite) vs. reducers** (`Annotated[..., operator.add]`) — why parallel writes to *different* keys are safe, and what you'd need for parallel writes to the *same* key.
- **Workflow patterns as fixed structures** — routing, parallelization, and (from your own prior work) evaluator-optimizer loops are all instances of a small set of composable graph shapes, distinct from open-ended "agent" behavior.